# Trial-wise responses analysis

- [x] LMM: linear mixed-effects model
- [x] Pairwise comparisons with emmeans
- [x] Residual diagnostics
- [x] Acceptance: cumulative link mixed model

In [ ]:

# # Install once if necessary:
# install.packages(c(
#   "tidyverse",
#   "lme4",
#   "lmerTest",
#   "emmeans",
#   "performance",
#   "ordinal",
#   "patchwork",
#   "broom.mixed"
# ))

library(tidyverse)
library(lme4)
library(lmerTest)
library(emmeans)
library(performance)
library(ordinal)
library(patchwork)
library(broom.mixed)

also installing the dependencies ‘sys’, ‘bit’, ‘ps’, ‘sass’, ‘cachem’, ‘labeling’, ‘RColorBrewer’, ‘viridisLite’, ‘rappdirs’, ‘rematch’, ‘askpass’, ‘bit64’, ‘prettyunits’, ‘otel’, ‘processx’, ‘highr’, ‘xfun’, ‘yaml’, ‘bslib’, ‘fontawesome’, ‘jquerylib’, ‘tinytex’, ‘listenv’, ‘parallelly’, ‘backports’, ‘generics’, ‘memoise’, ‘blob’, ‘DBI’, ‘R6’, ‘tidyselect’, ‘withr’, ‘data.table’, ‘isoband’, ‘S7’, ‘scales’, ‘gargle’, ‘cellranger’, ‘curl’, ‘ids’, ‘rematch2’, ‘cpp11’, ‘pkgconfig’, ‘mime’, ‘openssl’, ‘timechange’, ‘systemfonts’, ‘textshaping’, ‘clipr’, ‘vroom’, ‘tzdb’, ‘progress’, ‘callr’, ‘fs’, ‘knitr’, ‘rmarkdown’, ‘selectr’, ‘stringi’, ‘rbibutils’, ‘future’, ‘globals’, ‘broom’, ‘conflicted’, ‘dbplyr’, ‘dplyr’, ‘dtplyr’, ‘forcats’, ‘ggplot2’, ‘googledrive’, ‘googlesheets4’, ‘haven’, ‘hms’, ‘httr’, ‘lubridate’, ‘magrittr’, ‘modelr’, ‘purrr’, ‘ragg’, ‘readr’, ‘readxl’, ‘reprex’, ‘rstudioapi’, ‘rvest’, ‘stringr’, ‘tibble’, ‘tidyr’, ‘xml2’, ‘Rdpack’, ‘minqa’, ‘nloptr’, ‘reformulas’, ‘Rcpp’,


The downloaded binary packages are in
	/var/folders/0s/26zkp_wj5yv24vxwq1xdtxxw0000gn/T//Rtmpka4i4M/downloaded_packages


── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.2.1     ✔ readr     2.2.0
✔ forcats   1.0.1     ✔ stringr   1.6.0
✔ ggplot2   4.0.3     ✔ tibble    3.3.1
✔ lubridate 1.9.5     ✔ tidyr     1.3.2
✔ purrr     1.2.2     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors
Loading required package: Matrix


Attaching package: ‘Matrix’


The following objects are masked from ‘package:tidyr’:

    expand, pack, unpack



Attaching package: ‘lmerTest’


The following object is masked from ‘package:lme4’:

    lmer


The following object is masked from ‘package:stats’:

    step


Welcome to emmeans.
Caution: You lose important information if you filter this package's results.
See '? untidy'


Attaching package: ‘ordinal’


The following object

In [5]:
# Response variables analysed using LMM
li_resp_LMM <- c("SoPA", "SoNA", "SoA")
# Single ordinal response analysed using CLMM
li_resp_CLMM <- c("Accept")

long <- read_csv(
  "data_analysis/trial_wise_SoA_cleaned_long.csv",
  show_col_types = FALSE
)
wide <- read_csv(
  "data_analysis/trial_wise_SoA_cleaned_wide.csv",
  show_col_types = FALSE
)

# Explicitly declare categorical variables
long <- long %>%
  mutate(
    pp = factor(pp),
    condition = factor(condition),
    voice_id = factor(voice_id),
    response_type = factor(response_type)
  )

# main LMM
SoA ~ condition * voice_id + (1 | pp)

In [17]:
main_models <- list()
main_pairwise <- list()
main_variances <- list()

for (resp in li_resp_LMM) {

  cat("\n\n==================================================\n")
  cat("Outcome:", resp, "\n")
  cat("Model: condition × voice + participant intercept\n")
  cat("==================================================\n")

  model_formula <- as.formula(
    paste0(
      resp,
      " ~ condition * voice_id + (1 | pp)"
    )
  )

  # fit the model
  model <- lmer(
    formula = model_formula,
    data = wide,
    REML = TRUE,
  )
  main_models[[resp]] <- model
  cat("\nModel summary:\n")
  print(summary(model))

  # visualize the model
  tdat <- data.frame(predicted = predict(model),
      residual = residuals(model),
      referrer=wide$voice_id
      )
  ggplot(tdat,aes(x=condition,y=residual, colour=voice_id)) + geom_point() + geom_hline(yintercept=0, lty=3)

  # Condition comparisons averaged across voice
  emmeans(
    model,
    pairwise ~ condition,
    adjust = "holm"
  )

  # Voice comparison averaged across condition
  emmeans(
    model,
    pairwise ~ voice_id,
    adjust = "holm"
  )

  performance::check_model(model)

  cat("\nSingular fit:", isSingular(model), "\n")
}



Outcome: SoPA 
Model: condition × voice + participant intercept

Model summary:
Linear mixed model fit by REML. t-tests use Satterthwaite's method [
lmerModLmerTest]
Formula: model_formula
   Data: wide

REML criterion at convergence: 1077.7

Scaled residuals: 
    Min      1Q  Median      3Q     Max 
-3.1467 -0.5608 -0.0160  0.6063  4.0624 

Random effects:
 Groups   Name        Variance Std.Dev.
 pp       (Intercept) 0.7226   0.850   
 Residual             1.3328   1.154   
Number of obs: 326, groups:  pp, 28

Fixed effects:
                                  Estimate Std. Error        df t value
(Intercept)                        2.18929    0.22273  71.85794   9.830
conditionenhance                   1.03929    0.21818 293.21777   4.764
conditionrepeat                    2.14785    0.21926 293.33032   9.796
voice_idrobotic                   -0.02143    0.21818 293.21777  -0.098
conditionenhance:voice_idrobotic   0.25284    0.31173 293.37254   0.811
conditionrepeat:voice_idrobotic  

NOTE: Results may be misleading due to involvement in interactions

NOTE: Results may be misleading due to involvement in interactions




Singular fit: FALSE 


Outcome: SoNA 
Model: condition × voice + participant intercept

Model summary:
Linear mixed model fit by REML. t-tests use Satterthwaite's method [
lmerModLmerTest]
Formula: model_formula
   Data: wide

REML criterion at convergence: 1119.7

Scaled residuals: 
     Min       1Q   Median       3Q      Max 
-2.52588 -0.69554  0.06174  0.66441  2.42478 

Random effects:
 Groups   Name        Variance Std.Dev.
 pp       (Intercept) 0.4635   0.6808  
 Residual             1.5866   1.2596  
Number of obs: 326, groups:  pp, 28

Fixed effects:
                                 Estimate Std. Error       df t value Pr(>|t|)
(Intercept)                        4.8125     0.2119 105.8925  22.716  < 2e-16
conditionenhance                  -0.7143     0.2380 292.7786  -3.001  0.00293
conditionrepeat                   -1.3848     0.2392 292.9547  -5.789 1.82e-08
voice_idrobotic                    0.3482     0.2380 292.7786   1.463  0.14458
conditionenhance:voice_idrobotic  -0.3

NOTE: Results may be misleading due to involvement in interactions

NOTE: Results may be misleading due to involvement in interactions




Singular fit: FALSE 


Outcome: SoA 
Model: condition × voice + participant intercept

Model summary:
Linear mixed model fit by REML. t-tests use Satterthwaite's method [
lmerModLmerTest]
Formula: model_formula
   Data: wide

REML criterion at convergence: 1017.5

Scaled residuals: 
     Min       1Q   Median       3Q      Max 
-3.04094 -0.69399 -0.00548  0.61154  2.88627 

Random effects:
 Groups   Name        Variance Std.Dev.
 pp       (Intercept) 0.5878   0.7666  
 Residual             1.1060   1.0517  
Number of obs: 326, groups:  pp, 28

Fixed effects:
                                 Estimate Std. Error       df t value Pr(>|t|)
(Intercept)                        2.4745     0.2018  72.7027  12.259  < 2e-16
conditionenhance                   0.9464     0.1987 293.1958   4.762 3.02e-06
conditionrepeat                    1.9306     0.1997 293.3102   9.666  < 2e-16
voice_idrobotic                   -0.1148     0.1987 293.1958  -0.578    0.564
conditionenhance:voice_idrobotic   0.27

NOTE: Results may be misleading due to involvement in interactions

NOTE: Results may be misleading due to involvement in interactions




Singular fit: FALSE 


In [18]:
tdat <- data.frame(predicted = predict(model),
      residual = residuals(model),
      referrer=wide$voice_id
      )
  ggplot(tdat,aes(x=condition,y=residual, colour=voice_id)) + geom_point() + geom_hline(yintercept=0, lty=3)


ERROR while rich displaying an object: Error in `geom_point()`:
! Problem while computing aesthetics.
ℹ Error occurred in the 1st layer.
Caused by error:
! object 'condition' not found

Traceback:
1. sapply(x, f, simplify = simplify)
2. lapply(X = X, FUN = FUN, ...)
3. FUN(X[[i]], ...)
4. tryCatch(withCallingHandlers({
 .     if (!mime %in% names(repr::mime2repr)) 
 .         stop("No repr_* for mimetype ", mime, " in repr::mime2repr")
 .     rpr <- repr::mime2repr[[mime]](obj)
 .     if (is.null(rpr)) 
 .         return(NULL)
 .     prepare_content(is.raw(rpr), rpr)
 . }, error = error_handler), error = outer_handler)
5. tryCatchList(expr, classes, parentenv, handlers)
6. tryCatchOne(expr, names, parentenv, handlers[[1L]])
7. doTryCatch(return(expr), name, parentenv, handler)
8. withCallingHandlers({
 .     if (!mime %in% names(repr::mime2repr)) 
 .         stop("No repr_* for mimetype ", mime, " in repr::mime2repr")
 .     rpr <- repr::mime2repr[[mime]](obj)
 .     if (is.null(rpr)) 

In [19]:
model

Linear mixed model fit by REML ['lmerModLmerTest']
Formula: model_formula
   Data: wide
REML criterion at convergence: 1017.51
Random effects:
 Groups   Name        Std.Dev.
 pp       (Intercept) 0.7666  
 Residual             1.0517  
Number of obs: 326, groups:  pp, 28
Fixed Effects:
                     (Intercept)                  conditionenhance  
                          2.4745                            0.9464  
                 conditionrepeat                   voice_idrobotic  
                          1.9306                           -0.1148  
conditionenhance:voice_idrobotic   conditionrepeat:voice_idrobotic  
                          0.2766                            0.2316  